# LUNG-CANC'AIR — Notebook 6 : Tests de robustesse

**Objectif :** vérifier si le résultat du notebook 5 (mutations NF proches des sites Seveso SH)
tient toujours quand on creuse un peu, ou si c'est un effet fragile / un artefact.

**6 vérifications, dans l'ordre où les faire :**

| # | Question simple | Partie |
|---|---|---|
| 1 | Est-ce juste un effet "ville vs campagne" ? | Partie 1 |
| 2 | Est-ce que ça dépend du rayon choisi (3km) ? | Partie 2 |
| 3 | Est-ce porté par 1-2 usines seulement ? | Partie 3 |
| 4 | A-t-on testé trop de choses en même temps ? | Partie 4 |
| 5 | Est-ce que ça tient avec une définition de groupe différente ? | Partie 5 |
| 6 | Est-ce cohérent avec l'autre analyse du projet ? | Partie 6 |

⚠️ **Ce notebook suppose que le notebook `5-analyse_autocorrelation` a déjà tourné une fois**
dans le même environnement (même module `moran_spatial_analysis.py`). S'il n'a pas tourné,
la Partie 0 ci-dessous recharge tout depuis zéro.

## PARTIE 0 — Configuration et chargement

Si vous avez déjà exécuté le notebook 5 dans cette session, les cellules ci-dessous
détectent que `df`, `W`, `icpe_sh` existent déjà et ne rechargent rien (gain de temps).
Sinon elles refont le chargement à l'identique.

In [ ]:
import sys, os, importlib
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import geopandas as gpd

sys.path.append(r"H:\PFE Loice\Notebooks\Loice_Canc-air_2025\loice_pneumodetect\pneumodetect\src")
import moran_spatial_analysis as moran
importlib.reload(moran)

# =============================================================================
# ★ CONFIGURATION — reprend celle du notebook 5 + nouveaux paramètres ★
# =============================================================================
CONFIG = {
    # ── Chemins (identiques au notebook 5) ──────────────────────────────────
    'patients_csv' : r"H:\PFE Loice\Notebooks\Loice_Canc-air_2025\loice_pneumodetect\Data\patients_geocoded_clean_idf_2018_2023.csv",
    'icpe_shp'     : r"H:\PFE Loice\Notebooks\Loice_Canc-air_2025\loice_pneumodetect\Data\industries_a_risques\icpe.geojson\icpe_idf.shp",
    'output_dir'   : r"H:\PFE Loice\Notebooks\Loice_Canc-air_2025\loice_pneumodetect\Notebooks\output\resultats_robustesse",

    # ── NOUVEAU : IRIS — DEUX fichiers séparés à assembler ──────────────────
    # 1) le shapefile = la géométrie (les contours/polygones des IRIS)
    # 2) le csv       = la population (table INSEE "base population par IRIS")
    'iris_shp'         : r"À_COMPLETER\contours_iris_idf.shp",
    'iris_pop_csv'     : r"À_COMPLETER\population_iris_idf.csv",
    'iris_csv_sep'     : ';',        # les fichiers INSEE sont souvent en ';'
    'iris_csv_encoding': 'latin1',   # et en latin1/cp1252, pas en utf-8

    # ── Noms de colonnes à VÉRIFIER avec la cellule de diagnostic ci-dessous ──
    'col_pop'          : 'P21_POP',     # ← colonne "Population" totale dans le csv (à confirmer)
    'col_iris_code_csv': 'IRIS',        # ← colonne code IRIS dans le csv (à confirmer, parfois 'DCOMIRIS')
    'col_iris_code_shp': 'CODE_IRIS',   # ← colonne code IRIS dans le shapefile (à confirmer, parfois 'DCOMIRIS')

    # ── Paramètres spatiaux (identiques au notebook 5) ──────────────────────
    'rayon_m'        : 3000,
    'n_permutations' : 999,
    'alpha'          : 0.05,

    # ── NOUVEAU : rayons à tester pour la Partie 2 (sensibilité) ────────────
    'rayons_test_m'  : [1000, 2000, 3000, 5000, 10000],

    # ── Colonnes patients (identiques au notebook 5) ────────────────────────
    'col_x'      : 'x',
    'col_y'      : 'y',
    'col_id'     : 'pseudo_provisoire',
    'col_tabac'  : 'paquet_annee',

    # ── NOUVEAU : noms de colonnes (vérifiés avec votre fichier patients) ──
    'col_age'    : 'age_diagnostic',
    'col_sexe'   : 'sexe',

    'mutations_nf' : ['EGFR', 'ALK', 'ROS1', 'RET', 'NTRK', 'ERBB2', 'MET'],
}

print("✅ Configuration (notebook 6) définie")


In [ ]:
# Recharge seulement si nécessaire (évite de tout refaire si le notebook 5 a déjà tourné)
if 'df' not in dir() or 'W' not in dir():
    print("→ Rechargement complet (notebook 5 non détecté dans la session)...")
    df, gdf = moran.charger_patients(CONFIG)
    icpe, icpe_sh, icpe_sb, icpe_ns = moran.charger_icpe(CONFIG)
    df = moran.calculer_expositions_icpe(df, icpe_sh, icpe_sb, CONFIG['rayon_m'])
    W, n_voisins = moran.construire_matrice_poids(df, rayon_m=CONFIG['rayon_m'],
                                                   n_permutations=CONFIG['n_permutations'])
    print("✅ Données et matrice W (3km) reconstruites")
else:
    print("✅ Session existante détectée : df, gdf, icpe_sh, W réutilisés tels quels")

print(f"   Patients : {len(df)} | Groupe A (NF) : {df['groupe_A'].sum()}")


## PARTIE 1 — Est-ce juste un effet "ville vs campagne" ?

**L'idée en clair :** les usines Seveso sont presque toujours construites loin des
centres-villes, parce qu'il faut de la place. Si les patients mutés NF habitent
*aussi*, par ailleurs, plutôt en zone moins peuplée — alors le résultat du notebook 5
ne dit peut-être rien sur la pollution : il dit juste "les gens qui habitent loin
de la ville habitent loin de la ville".

**Ce qu'on fait ici :**
1. On assemble vos deux fichiers : le shapefile (contours des IRIS) + le csv (population par IRIS), reliés par le code IRIS.
2. On calcule la densité = population / surface pour chaque IRIS, puis on l'attribue à chaque patient (le quartier dans lequel il habite).
3. On regarde si les patients NF habitent des quartiers moins denses que les autres.
4. Si oui → on refait le test de Moran bivarié **séparément dans des quartiers de densité comparable**.
   Si le signal Seveso×NF disparaît une fois qu'on compare des quartiers similaires, c'est
   que la densité expliquait (au moins en partie) le résultat du notebook 5.
5. On ajoute aussi une régression logistique simple qui met toutes les variables
   ensemble (densité, âge, sexe, tabac) pour voir si l'exposition Seveso reste liée
   à la mutation NF une fois ces facteurs neutralisés.

In [ ]:
# ── 1a-0. Diagnostic : vérifier les vrais noms de colonnes avant de continuer ──
# (le dictionnaire de variables INSEE donne des DESCRIPTIONS, pas les en-têtes réelles
#  du csv -- on les affiche ici pour pouvoir corriger CONFIG ci-dessus si besoin)

pop_diag = pd.read_csv(CONFIG['iris_pop_csv'], sep=CONFIG['iris_csv_sep'],
                       encoding=CONFIG['iris_csv_encoding'], nrows=5)
print("Colonnes du CSV population :")
print(pop_diag.columns.tolist())
print()
print(pop_diag.head())

iris_diag = gpd.read_file(CONFIG['iris_shp'], rows=5)
print("\nColonnes du shapefile IRIS :")
print(iris_diag.columns.tolist())

print('''
→ Repérez dans la liste ci-dessus :
   - la colonne "Population" totale (PAS une tranche d'âge) -> CONFIG['col_pop']
   - la colonne code IRIS du csv                            -> CONFIG['col_iris_code_csv']
   - la colonne code IRIS du shapefile                      -> CONFIG['col_iris_code_shp']
  Corrigez CONFIG si les noms affichés ne correspondent pas, puis relancez depuis la cellule
  de configuration (Partie 0) avant de continuer.
''')


In [ ]:
# ── 1a. Chargement et assemblage : géométrie (shp) + population (csv) ───────
try:
    iris_geo = gpd.read_file(CONFIG['iris_shp'])
    iris_geo = iris_geo.to_crs(epsg=2154)  # Lambert 93, même CRS que les patients

    iris_pop = pd.read_csv(CONFIG['iris_pop_csv'], sep=CONFIG['iris_csv_sep'],
                            encoding=CONFIG['iris_csv_encoding'])

    # Les codes IRIS sont du texte (ex: "751010001") -> on force le typage pour éviter
    # les ratés de jointure dus à des zéros de tête perdus en lisant comme un nombre.
    iris_geo[CONFIG['col_iris_code_shp']] = iris_geo[CONFIG['col_iris_code_shp']].astype(str).str.zfill(9)
    iris_pop[CONFIG['col_iris_code_csv']] = iris_pop[CONFIG['col_iris_code_csv']].astype(str).str.zfill(9)

    iris = iris_geo.merge(
        iris_pop[[CONFIG['col_iris_code_csv'], CONFIG['col_pop']]],
        left_on=CONFIG['col_iris_code_shp'], right_on=CONFIG['col_iris_code_csv'],
        how='left'
    )

    n_sans_pop = iris[CONFIG['col_pop']].isna().sum()
    print(f"✅ Jointure géométrie + population : {len(iris) - n_sans_pop}/{len(iris)} IRIS avec une population")
    if n_sans_pop > 0:
        print(f"   ⚠️  {n_sans_pop} IRIS sans population assignée (codes qui ne matchent pas -> à vérifier)")

    # Surface en km² (la géométrie est en mètres -> Lambert93 convient)
    iris['surface_km2'] = iris.geometry.area / 1e6
    iris['densite_pop'] = iris[CONFIG['col_pop']] / iris['surface_km2']

    print(f"   Densité médiane : {iris['densite_pop'].median():.0f} hab/km²")
    iris_ok = True
except Exception as e:
    print("⚠️  Erreur lors du chargement/assemblage IRIS.")
    print(f"   Erreur : {e}")
    print("   → Relancez d'abord la cellule de diagnostic ci-dessus pour vérifier les noms de colonnes.")
    iris_ok = False


In [ ]:
# ── 1b. Rapprochement patient -> IRIS par CODE_IRIS (pas de jointure spatiale nécessaire) ──
# Bonne nouvelle : votre fichier patients contient déjà une colonne CODE_IRIS
# (le géocodage avait déjà été poussé jusqu'au niveau IRIS). On fait donc un simple
# rapprochement par code, plus rapide et plus fiable qu'une jointure spatiale
# "le point tombe-t-il dans ce polygone" (qui peut rater sur les points pile à la frontière).

if iris_ok:
    # On retire d'abord toute colonne 'densite_pop' déjà présente (utile si vous
    # relancez cette cellule plusieurs fois -- sinon le 2e merge crée densite_pop_x/_y
    # au lieu de densite_pop, et la cellule suivante plante avec un KeyError)
    df = df.drop(columns=[c for c in ['densite_pop'] if c in df.columns])

    df['CODE_IRIS'] = df['CODE_IRIS'].astype(str).str.strip().str.zfill(9)

    df = df.merge(
        iris[[CONFIG['col_iris_code_shp'], 'densite_pop']].rename(
            columns={CONFIG['col_iris_code_shp']: 'CODE_IRIS'}
        ),
        on='CODE_IRIS', how='left'
    )

    n_manquants = df['densite_pop'].isna().sum()
    print(f"✅ Densité de population assignée à {len(df) - n_manquants}/{len(df)} patients (jointure par CODE_IRIS)")
    if n_manquants > 0:
        print(f"   ⚠️  {n_manquants} patients sans correspondance (format de code IRIS différent entre les 2 fichiers ?)")


In [ ]:
# ── 1c. Les patients NF habitent-ils des zones moins denses ? ────────────────
if iris_ok:
    from scipy.stats import mannwhitneyu

    d_A = df.loc[df['groupe_A'] == 1, 'densite_pop'].dropna()
    d_B = df.loc[df['groupe_A'] == 0, 'densite_pop'].dropna()
    _, p_dens = mannwhitneyu(d_A, d_B, alternative='two-sided')

    print("── Densité de population : Groupe A (mutations NF) vs reste ──")
    print(f"  médiane NF   = {d_A.median():.0f} hab/km²")
    print(f"  médiane reste= {d_B.median():.0f} hab/km²")
    print(f"  p = {'<0.001' if p_dens < 0.001 else f'{p_dens:.3f}'} "
          f"{'✅ différence significative' if p_dens < CONFIG['alpha'] else '— pas de différence significative'}")

    fig, ax = plt.subplots(figsize=(5, 5))
    bp = ax.boxplot([d_A, d_B], labels=['Groupe A\n(mut. NF)', 'Reste'],
                     patch_artist=True, widths=0.5, showfliers=False)
    bp['boxes'][0].set_facecolor('#2E86AB'); bp['boxes'][0].set_alpha(0.7)
    bp['boxes'][1].set_facecolor('#AAAAAA'); bp['boxes'][1].set_alpha(0.7)
    ax.set_ylabel('Densité de population (hab/km²)')
    ax.set_title('Densité du quartier de résidence', fontweight='bold')
    ax.grid(True, axis='y', alpha=0.3)
    plt.tight_layout(); plt.show()


In [ ]:
# ── 1d. Test de Moran bivarié SÉPARÉMENT par strate de densité ──────────────
# Idée : si le signal Seveso x NF survit DANS CHAQUE strate (donc en comparant des
# patients qui habitent des zones de densité comparable), c'est plus solide.
# S'il disparaît dans chaque strate, c'est que la densité expliquait le résultat global.

if iris_ok:
    df['strate_densite'] = pd.qcut(df['densite_pop'], q=3, labels=['Rural/périurbain', 'Intermédiaire', 'Urbain dense'])

    print("Répartition des strates :")
    print(df['strate_densite'].value_counts())
    print()

    resultats_strates = []
    for strate in ['Rural/périurbain', 'Intermédiaire', 'Urbain dense']:
        sous_df = df[df['strate_densite'] == strate].reset_index(drop=True)
        if sous_df['groupe_A'].sum() < 10:
            print(f"⚠️  Strate '{strate}' : trop peu de patients mutés NF ({sous_df['groupe_A'].sum()}), test ignoré")
            continue

        W_strate, _ = moran.construire_matrice_poids(sous_df, rayon_m=CONFIG['rayon_m'],
                                                       n_permutations=CONFIG['n_permutations'])
        res_strate = moran.moran_bivarie(
            sous_df, W_strate,
            var_x='groupe_A', var_y='score_expo_SH',
            label_x='Mutations NF', label_y=f'Score expo SH ({strate})',
            n_perm=CONFIG['n_permutations']
        )
        resultats_strates.append({
            'strate': strate, 'n': len(sous_df), 'n_NF': sous_df['groupe_A'].sum(),
            'I': res_strate['I'], 'p_sim': res_strate['p_sim'],
            'sig': '✅' if res_strate['p_sim'] < CONFIG['alpha'] else '—'
        })

    df_strates = pd.DataFrame(resultats_strates)
    print("\n── Résultat du Moran bivarié, strate par strate ──")
    print(df_strates.to_string(index=False))
    print("\n→ Si 'sig' reste ✅ dans chaque strate : le signal ne s'explique pas par la densité seule.")
    print("→ Si 'sig' devient '—' partout : la densité explique probablement le résultat global.")


In [ ]:
# ── 1e. Régression logistique ajustée (vue individuelle, pas spatiale) ──────
# On regarde si l'exposition Seveso est encore liée à la mutation NF une fois qu'on
# tient compte en même temps de la densité, de l'âge, du sexe et du tabac.
# (Ce n'est pas un modèle spatial type SPDE-INLA — c'est une première étape simple
#  et rapide à faire avant d'aller vers un modèle plus lourd.)

if iris_ok:
    import statsmodels.api as sm
    import statsmodels.formula.api as smf

    colonnes_modele = ['groupe_A', 'score_expo_SH', 'densite_pop', CONFIG['col_tabac']]
    for c in [CONFIG['col_age'], CONFIG['col_sexe']]:
        if c in df.columns:
            colonnes_modele.append(c)
        else:
            print(f"⚠️  Colonne '{c}' absente de df — adaptez CONFIG['col_age']/CONFIG['col_sexe']. Variable ignorée.")

    df_modele = df[colonnes_modele].dropna()
    formule = f"groupe_A ~ score_expo_SH + densite_pop + {CONFIG['col_tabac']}"
    if CONFIG['col_age'] in df_modele.columns:
        formule += f" + {CONFIG['col_age']}"
    if CONFIG['col_sexe'] in df_modele.columns:
        formule += f" + C({CONFIG['col_sexe']})"

    print(f"Modèle ajusté : {formule}\n")
    modele = smf.logit(formule, data=df_modele).fit(disp=0)
    print(modele.summary())

    print("\n→ Regardez la ligne 'score_expo_SH' : si son p reste < 0.05 ET son coefficient")
    print("   garde le même signe qu'avant ajustement, l'association résiste à l'ajustement.")


**📊 À noter dans le rapport (Partie 1) :** *(à compléter après exécution)*
- Densité médiane Groupe A vs reste : …
- Le signal Seveso×NF tient-il dans chaque strate de densité ? …
- Le score d'exposition reste-t-il significatif après ajustement ? …

## PARTIE 2 — Est-ce que ça dépend du rayon choisi (3 km) ?

**L'idée en clair :** le notebook 5 a tout calculé avec un rayon fixe de 3 km. C'est un
choix arbitraire. Si le résultat n'existe qu'à exactement 3 km et disparaît à 2 km ou
5 km, c'est mauvais signe — ça voudrait dire que c'est un hasard lié à ce chiffre
précis, pas un vrai phénomène géographique.

**Ce qu'on fait ici :** on refait exactement le même test (Moran bivarié NF × Seveso)
avec plusieurs rayons : 1, 2, 3, 5 et 10 km, et on regarde si le résultat reste stable.

In [ ]:
resultats_rayons = []

for r in CONFIG['rayons_test_m']:
    print(f"→ Rayon {r/1000:.0f} km...")
    df_r = df.copy()
    df_r = moran.calculer_expositions_icpe(df_r, icpe_sh, icpe_sb, r)
    W_r, n_voisins_r = moran.construire_matrice_poids(df_r, rayon_m=r,
                                                       n_permutations=CONFIG['n_permutations'])
    res_r = moran.moran_bivarie(
        df_r, W_r,
        var_x='groupe_A', var_y='score_expo_SH',
        label_x='Mutations NF', label_y='Score expo SH',
        n_perm=CONFIG['n_permutations']
    )
    resultats_rayons.append({
        'rayon_km': r / 1000,
        'I': res_r['I'],
        'p_sim': res_r['p_sim'],
        'sig': '✅' if res_r['p_sim'] < CONFIG['alpha'] else '—',
        'isoles': sum(1 for n in n_voisins_r if n == 0)
    })

df_rayons = pd.DataFrame(resultats_rayons)
print("\n── Sensibilité au rayon ──")
print(df_rayons.to_string(index=False))


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

axes[0].plot(df_rayons['rayon_km'], df_rayons['I'], 'o-', color='#2E86AB', linewidth=2)
axes[0].axhline(0, color='grey', linestyle='--', alpha=0.5)
axes[0].set_xlabel('Rayon (km)'); axes[0].set_ylabel("Indice de Moran bivarié (I)")
axes[0].set_title("L'effet selon le rayon choisi", fontweight='bold')
axes[0].grid(True, alpha=0.3)

axes[1].plot(df_rayons['rayon_km'], df_rayons['p_sim'], 'o-', color='#E76F51', linewidth=2)
axes[1].axhline(CONFIG['alpha'], color='red', linestyle='--', label=f"seuil α={CONFIG['alpha']}")
axes[1].set_xlabel('Rayon (km)'); axes[1].set_ylabel('p-value (permutation)')
axes[1].set_title('Significativité selon le rayon', fontweight='bold')
axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.tight_layout(); plt.show()

print("→ Si les points restent sous la ligne rouge sur plusieurs rayons consécutifs,")
print("  le résultat est robuste. S'il n'est sous la ligne qu'à un seul rayon (3km),")
print("  c'est un signal fragile, probablement un coup de chance statistique.")


**📊 À noter dans le rapport (Partie 2) :** *(à compléter après exécution)*
- Le résultat reste-t-il significatif sur au moins 2-3 rayons consécutifs ? …
- Rayon où l'effet est le plus net : …

## PARTIE 3 — Est-ce porté par 1 ou 2 usines seulement ?

**L'idée en clair :** il n'y a que 37 sites Seveso "Seuil Haut" en Île-de-France pour
1682 patients. Le résultat du notebook 5 ("64 patients en co-cluster") pourrait en
réalité être tiré par un tout petit nombre de sites, voire un seul. Si c'est le cas, le
résultat n'est pas un phénomène général en Île-de-France, mais une histoire très locale
(une commune, un quartier) — ce qui se présente et s'interprète très différemment.

**Ce qu'on fait ici :** pour chacun des 64 patients "co-cluster" (HH), on identifie le
site Seveso SH le plus proche, et on compte combien de patients sont rattachés à chaque
site.

In [ ]:
from scipy.spatial import cKDTree

# On recalcule le LISA bivarié à 3km pour récupérer la liste des patients HH
# (si déjà calculé dans le notebook 5 et présent en mémoire, ceci les recalcule à l'identique)
df, lisa_bv_A, clusters_bv_A = moran.lisa_bivarie(
    df, W,
    var_x='groupe_A', var_y='score_expo_SH',
    label_x='Mutations NF', label_y='Score expo SH',
    icpe_sh=icpe_sh,
    n_perm=CONFIG['n_permutations'], alpha=CONFIG['alpha']
)

patients_hh = df[df['lisa_bv_type'] == 'HH'].copy()
print(f"Patients en co-cluster HH : {len(patients_hh)}")

# Coordonnées des sites Seveso SH (en Lambert93, comme les patients)
coords_sh = np.column_stack([icpe_sh.geometry.x, icpe_sh.geometry.y])
arbre = cKDTree(coords_sh)

coords_patients = patients_hh[['x_l93', 'y_l93']].values
dist_proche, idx_proche = arbre.query(coords_patients)

patients_hh['site_id_proche'] = idx_proche
patients_hh['dist_site_proche_m'] = dist_proche


In [ ]:
comptage_sites = patients_hh['site_id_proche'].value_counts().sort_values(ascending=False)
n_sites_distincts = len(comptage_sites)

print(f"Nombre de sites Seveso distincts concernés : {n_sites_distincts} (sur 37 au total)")
print()
print("── Répartition des 64 patients HH par site le plus proche ──")
print(comptage_sites)

part_top3 = comptage_sites.head(3).sum() / len(patients_hh) * 100
print(f"\n→ Les 3 sites les plus représentés concentrent {part_top3:.0f}% des patients HH")
print("  (si ce chiffre est élevé, ex. >50%, le résultat global est surtout porté")
print("   par une poignée de sites/quartiers précis, pas par un phénomène diffus en IDF)")


In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
comptage_sites.plot(kind='bar', ax=ax, color='#E76F51', alpha=0.8)
ax.set_xlabel('Identifiant du site Seveso SH le plus proche')
ax.set_ylabel('Nombre de patients HH rattachés')
ax.set_title('Concentration des co-clusters autour des sites Seveso', fontweight='bold')
ax.grid(True, axis='y', alpha=0.3)
plt.tight_layout(); plt.show()


**📊 À noter dans le rapport (Partie 3) :** *(à compléter après exécution)*
- Nombre de sites distincts concernés : …
- Part portée par les 3 principaux sites : …
- Conclusion : phénomène diffus en IDF, ou concentré sur quelques sites/communes précis ? …

## PARTIE 4 — A-t-on testé trop de choses en même temps ?

**L'idée en clair :** dans le notebook 5, on a fait 7 tests de Moran différents. Plus on
fait de tests, plus le risque de trouver un résultat "significatif" par pur hasard
augmente — un peu comme lancer une pièce 7 fois et s'étonner d'avoir eu "face" au moins
une fois.

**Ce qu'on fait ici :** on applique deux corrections statistiques classiques
(Bonferroni, plus stricte ; et Benjamini-Hochberg/FDR, plus souple) aux 7 p-values du
notebook 5, et on regarde lesquelles restent significatives après correction.

In [ ]:
# On recalcule les 7 tests du notebook 5 pour être sûr d'avoir les bonnes p-values
# (si déjà en mémoire depuis le notebook 5, ceci donne exactement les mêmes valeurs)

res_moran_A      = moran.moran_global_univarie(df, W, variable='groupe_A',
                       label='Mutations NF (Groupe A)', couleur='#2E86AB', n_perm=CONFIG['n_permutations'])
res_moran_C      = moran.moran_global_univarie(df, W, variable='groupe_C',
                       label='Non-fumeurs (Groupe C)', couleur='#52B788', n_perm=CONFIG['n_permutations'])
res_moran_SH     = moran.moran_global_univarie(df, W, variable='score_expo_SH',
                       label='Score exposition Seveso SH', couleur='#E76F51', n_perm=CONFIG['n_permutations'])
res_moran_nb     = moran.moran_global_univarie(df, W, variable='nb_SH_3km',
                       label='Nb Seveso SH dans 3km', couleur='#7B2D8B', n_perm=CONFIG['n_permutations'])
res_bv_A_score   = moran.moran_bivarie(df, W, var_x='groupe_A', var_y='score_expo_SH',
                       label_x='Mutations NF', label_y='Score expo Seveso SH', n_perm=CONFIG['n_permutations'])
res_bv_A_nb      = moran.moran_bivarie(df, W, var_x='groupe_A', var_y='nb_SH_3km',
                       label_x='Mutations NF', label_y='Nb SH 3km', n_perm=CONFIG['n_permutations'])
res_bv_AC_score  = moran.moran_bivarie(df, W, var_x='groupe_AC', var_y='score_expo_SH',
                       label_x='Mutations NF + Non-fumeurs (A+C)', label_y='Score expo Seveso SH',
                       n_perm=CONFIG['n_permutations'])

print("✅ Les 7 tests du notebook 5 ont été recalculés")


In [ ]:
from statsmodels.stats.multitest import multipletests

tests = [
    ("Moran global — Mutations NF",                    res_moran_A['p_sim']),
    ("Moran global — Non-fumeurs",                      res_moran_C['p_sim']),
    ("Moran global — Score expo Seveso SH",             res_moran_SH['p_sim']),
    ("Moran global — Nb Seveso SH 3km",                 res_moran_nb['p_sim']),
    ("Moran bivarié — NF × Score expo SH",              res_bv_A_score['p_sim']),
    ("Moran bivarié — NF × Nb SH 3km",                  res_bv_A_nb['p_sim']),
    ("Moran bivarié — (NF+non-fum) × Score expo SH",    res_bv_AC_score['p_sim']),
]

labels = [t[0] for t in tests]
p_bruts = [t[1] for t in tests]

sig_bonf, p_bonf, _, _ = multipletests(p_bruts, alpha=CONFIG['alpha'], method='bonferroni')
sig_fdr,  p_fdr,  _, _ = multipletests(p_bruts, alpha=CONFIG['alpha'], method='fdr_bh')

df_correction = pd.DataFrame({
    'Test': labels,
    'p brut': p_bruts,
    'Sig. brut (α=0.05)': ['✅' if p < CONFIG['alpha'] else '—' for p in p_bruts],
    'p Bonferroni': p_bonf.round(4),
    'Sig. Bonferroni': ['✅' if s else '—' for s in sig_bonf],
    'p FDR (Benjamini-Hochberg)': p_fdr.round(4),
    'Sig. FDR': ['✅' if s else '—' for s in sig_fdr],
})

print("── Effet de la correction pour tests multiples ──\n")
print(df_correction.to_string(index=False))

print("\n→ Un test qui reste ✅ après Bonferroni est solide.")
print("→ Un test ✅ seulement avant correction est plus fragile : à présenter avec prudence.")


**📊 À noter dans le rapport (Partie 4) :** *(à compléter après exécution)*
- Quels tests restent significatifs après Bonferroni ? …
- Quels tests restent significatifs seulement après FDR (plus souple) ? …
- Quels tests ne survivent à aucune correction ? …

## PARTIE 5 — Est-ce que ça tient avec une définition de groupe différente ?

**L'idée en clair :** le notebook 5 a montré que le résultat tient pour le "Groupe A"
(7 mutations regroupées) mais disparaît si on l'élargit au "Groupe A+C" (en ajoutant les
non-fumeurs sans mutation). C'est un signal d'alerte à creuser : si le résultat bouge
beaucoup selon la définition exacte du groupe, il faut le dire clairement plutôt que de
ne montrer que la version qui "marche".

**Ce qu'on fait ici :** on teste d'autres découpages plausibles (une mutation à la fois,
par exemple) et on regarde si le signal Seveso×mutation reste cohérent. Vos colonnes de
mutations s'appellent `mutation_EGFR`, `mutation_ALK`, etc. (confirmé avec votre fichier
patients) — le code ci-dessous utilise déjà ce préfixe.

In [ ]:
colonnes_a_tester = ['mutation_' + m for m in CONFIG['mutations_nf']]
# -> ['mutation_EGFR', 'mutation_ALK', 'mutation_ROS1', 'mutation_RET',
#     'mutation_NTRK', 'mutation_ERBB2', 'mutation_MET']

resultats_robustesse = []

for col in colonnes_a_tester:
    if col not in df.columns:
        print(f"⚠️  Colonne '{col}' absente de df — test ignoré (vérifiez le nom exact dans votre fichier patients)")
        continue
    n_pos = df[col].sum()
    if n_pos < 10:
        print(f"⚠️  '{col}' : seulement {n_pos} patients — trop peu pour un test fiable, ignoré")
        continue

    res_col = moran.moran_bivarie(
        df, W, var_x=col, var_y='score_expo_SH',
        label_x=col, label_y='Score expo Seveso SH',
        n_perm=CONFIG['n_permutations']
    )
    resultats_robustesse.append({
        'Mutation testée': col, 'n patients': int(n_pos),
        'I': res_col['I'], 'p_sim': res_col['p_sim'],
        'Sig.': '✅' if res_col['p_sim'] < CONFIG['alpha'] else '—'
    })

# On ajoute pour comparaison les deux groupes déjà testés dans le notebook 5
resultats_robustesse.append({
    'Mutation testée': 'Groupe A (7 mutations, notebook 5)', 'n patients': int(df['groupe_A'].sum()),
    'I': res_bv_A_score['I'], 'p_sim': res_bv_A_score['p_sim'],
    'Sig.': '✅' if res_bv_A_score['p_sim'] < CONFIG['alpha'] else '—'
})
resultats_robustesse.append({
    'Mutation testée': 'Groupe A+C (notebook 5)', 'n patients': int(df['groupe_AC'].sum()),
    'I': res_bv_AC_score['I'], 'p_sim': res_bv_AC_score['p_sim'],
    'Sig.': '✅' if res_bv_AC_score['p_sim'] < CONFIG['alpha'] else '—'
})

df_robustesse = pd.DataFrame(resultats_robustesse)
print("── Le signal Seveso × mutation, mutation par mutation ──\n")
print(df_robustesse.to_string(index=False))

print("\n→ Si plusieurs mutations isolées vont dans le même sens (I>0), c'est plus crédible.")
print("→ Si une seule mutation porte tout le signal du Groupe A, dites-le explicitement.")


**📊 À noter dans le rapport (Partie 5) :** *(à compléter après exécution)*
- Quelle(s) mutation(s) individuelle(s) montre(nt) un signal cohérent avec le Groupe A ? …
- Le signal du Groupe A est-il porté par une seule mutation majoritaire, ou partagé ? …

## PARTIE 6 — Est-ce cohérent avec l'autre analyse du projet ?

**L'idée en clair :** ce notebook regarde la géographie (Moran). Mais une autre analyse
du projet (régression logistique, document de méthodologie) a déjà regardé, patient par
patient, si les patients EGFR/NF habitent plus souvent près des sites ICPE-SH (sans
notion spatiale de "clustering", juste "proche ou pas"). Si les deux méthodes,
complètement différentes, vont dans le même sens, c'est un signal plus crédible que
chacune isolément.

**Ce qu'on fait ici :** rien à calculer dans ce notebook — c'est une vérification à
faire à la main en relisant les deux résultats côte à côte, une fois que les "OR = …"
du document de méthodologie seront complétés avec les vraies valeurs (ils sont encore
en attente de chiffrage dans le document RESULTS).

**📊 À compléter dans le rapport (Partie 6) — tableau de comparaison à remplir une fois les deux analyses finalisées :**

| | Régression logistique (individuelle) | Moran bivarié (spatial, ce notebook) |
|---|---|---|
| Sens de l'association | Plus proche des ICPE-SH chez NF (OR=…, p=0.052, à la limite) | Co-clustering positif (I=0.016-0.023, p=0.004-0.023) |
| Force du signal | Faible / à la limite | Faible |
| Robuste aux contrôles ? | À vérifier (ajustement âge/sexe/tabac déjà fait ?) | À vérifier (Parties 1, 4, 5 de ce notebook) |
| **Conclusion croisée** | *(à rédiger une fois les deux analyses stabilisées)* | |


## PARTIE 7 — Synthèse finale

Checklist à remplir une fois toutes les parties exécutées.

In [ ]:
print("═"*70)
print("SYNTHÈSE DES TESTS DE ROBUSTESSE")
print("═"*70)

print('''
1. Densité de population (Partie 1)
   [ ] Les NF habitent des zones moins denses : oui / non
   [ ] Le signal Seveso×NF tient dans chaque strate de densité : oui / non
   [ ] Le score d'exposition reste significatif après ajustement : oui / non

2. Sensibilité au rayon (Partie 2)
   [ ] Le signal est stable sur au moins 2-3 rayons consécutifs : oui / non

3. Concentration géographique (Partie 3)
   [ ] Nombre de sites distincts concernés : ___
   [ ] Part portée par les 3 principaux sites : ___%

4. Tests multiples (Partie 4)
   [ ] Tests survivant à Bonferroni : ___
   [ ] Tests survivant seulement au FDR : ___

5. Robustesse de la définition de groupe (Partie 5)
   [ ] Mutations individuelles cohérentes avec le Groupe A : ___

6. Cohérence avec l'analyse non-spatiale (Partie 6)
   [ ] Même sens d'association : oui / non / à vérifier

──────────────────────────────────────────────────────────────────
VERDICT GLOBAL (à rédiger une fois les ✅/⚠️ ci-dessus renseignés) :
──────────────────────────────────────────────────────────────────
''')


---
**Rappel méthodologique à garder en tête pour la rédaction finale :** même si toutes les
vérifications ci-dessus passent au vert, ce notebook reste de la statistique spatiale
descriptive (corrélation géographique). Cela ne prouve pas qu'habiter près d'un site
Seveso *cause* une mutation. Ça permet seulement de dire si le signal observé est solide,
ou s'il fond dès qu'on regarde d'un peu plus près.